# 01 - Load and inspect the Olist tables

**Goal:** understand every table before cleaning anything: size, data types, missing values, duplicates, keys and how the tables connect.

**Rule for this notebook:** after each section, write what you noticed in the Findings cell at the bottom, in your own words.

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can reuse code from src/
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.append(str(root))

import pandas as pd

from src.extract import load_raw_tables
from src.profiling import missing_keys, overview, profile_table

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

tables = load_raw_tables()

2026-11-08 20:42:52,712 | INFO | src.extract | Loaded customers                 99441 rows,  5 columns
2026-11-08 20:42:52,899 | INFO | src.extract | Loaded geolocation             1000163 rows,  5 columns
2026-11-08 20:42:53,014 | INFO | src.extract | Loaded order_items              112650 rows,  7 columns
2026-11-08 20:42:53,071 | INFO | src.extract | Loaded order_payments           103886 rows,  5 columns
2026-11-08 20:42:53,215 | INFO | src.extract | Loaded order_reviews             99224 rows,  7 columns
2026-11-08 20:42:53,379 | INFO | src.extract | Loaded orders                    99441 rows,  8 columns
2026-11-08 20:42:53,402 | INFO | src.extract | Loaded products                  32951 rows,  9 columns
2026-11-08 20:42:53,406 | INFO | src.extract | Loaded sellers                    3095 rows,  4 columns
2026-11-08 20:42:53,409 | INFO | src.extract | Loaded category_translation         71 rows,  2 columns


## 1. Overview of all tables
How big is each table, and does any table contain fully duplicated rows?

In [2]:
overview(tables)

,rows,columns,duplicate_rows,memory_mb
table,,,,
customers,99441,5,0,26.59
geolocation,1000163,5,261831,130.26
order_items,112650,7,0,35.99
order_payments,103886,5,0,16.23
order_reviews,99224,7,0,39.12
orders,99441,8,0,52.94
products,32951,9,0,6.30
sellers,3095,4,0,0.59
category_translation,71,2,0,0.01


## 2. Table by table: sample rows and column profile
For each table: look at 3 sample rows, then at each column's data type, null count and number of unique values.

In [3]:
for name, df in tables.items():
    print("=" * 80)
    print(name.upper(), df.shape)
    display(df.head(3))
    display(profile_table(df))

CUSTOMERS (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


,dtype,nulls,null_pct,unique
customer_id,str,0,0.0,99441
customer_unique_id,str,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,str,0,0.0,4119
customer_state,str,0,0.0,27


GEOLOCATION (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


,dtype,nulls,null_pct,unique
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717372
geolocation_lng,float64,0,0.0,717615
geolocation_city,str,0,0.0,8011
geolocation_state,str,0,0.0,27


... (17 outputs left out)

,dtype,nulls,null_pct,unique
seller_id,str,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,str,0,0.0,611
seller_state,str,0,0.0,23


CATEGORY_TRANSLATION (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


,dtype,nulls,null_pct,unique
product_category_name,str,0,0.0,71
product_category_name_english,str,0,0.0,71


## 3. Are the keys unique?
A key that should be unique but isn't will silently multiply rows in joins later, so check now.

In [4]:
orders = tables["orders"]
customers = tables["customers"]
items = tables["order_items"]
products = tables["products"]
sellers = tables["sellers"]
reviews = tables["order_reviews"]
payments = tables["order_payments"]
translation = tables["category_translation"]

key_checks = {
    "orders.order_id is unique": orders["order_id"].is_unique,
    "customers.customer_id is unique": customers["customer_id"].is_unique,
    "products.product_id is unique": products["product_id"].is_unique,
    "sellers.seller_id is unique": sellers["seller_id"].is_unique,
    "order_items (order_id, order_item_id) is unique": not items.duplicated(["order_id", "order_item_id"]).any(),
    "order_reviews.review_id is unique": reviews["review_id"].is_unique,
}
pd.Series(key_checks, name="passes")

orders.order_id is unique                           True
customers.customer_id is unique                     True
products.product_id is unique                       True
sellers.seller_id is unique                         True
order_items (order_id, order_item_id) is unique     True
order_reviews.review_id is unique                  False
Name: passes, dtype: bool

If any check above is `False`, investigate it in the next cell before moving on. Look at a few of the offending rows and decide what it means.

In [5]:
# Example investigation: edit this to look at whichever check failed
dupes = reviews[reviews.duplicated("review_id", keep=False)].sort_values("review_id")
print("rows with a repeated review_id:", len(dupes))
dupes.head(10)

rows with a repeated review_id: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


## 4. Do the tables connect? (orphan check)
Count child rows whose key does not exist in the parent table. Zero means the relationship is clean.

In [6]:
orphans = {
    "orders -> customers": missing_keys(orders, "customer_id", customers, "customer_id"),
    "order_items -> orders": missing_keys(items, "order_id", orders, "order_id"),
    "order_items -> products": missing_keys(items, "product_id", products, "product_id"),
    "order_items -> sellers": missing_keys(items, "seller_id", sellers, "seller_id"),
    "order_payments -> orders": missing_keys(payments, "order_id", orders, "order_id"),
    "order_reviews -> orders": missing_keys(reviews, "order_id", orders, "order_id"),
}
pd.Series(orphans, name="orphan_rows")

orders -> customers         0
order_items -> orders       0
order_items -> products     0
order_items -> sellers      0
order_payments -> orders    0
order_reviews -> orders     0
Name: orphan_rows, dtype: int64

## 5. Orders: status and dates

In [7]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
date_cols = [c for c in orders.columns if c.endswith(("_at", "_date", "_timestamp"))]
orders_dt = orders.copy()
for col in date_cols:
    orders_dt[col] = pd.to_datetime(orders_dt[col])

# Earliest and latest value of each date column
orders_dt[date_cols].agg(["min", "max"]).T

,min,max
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


In [9]:
# Delivered orders that have no delivery date: a data quality red flag
delivered_no_date = orders_dt[
    (orders_dt["order_status"] == "delivered") & orders_dt["order_delivered_customer_date"].isna()
]
print("delivered orders with no delivery date:", len(delivered_no_date))

delivered orders with no delivery date: 8


## 6. Product categories without an English translation

In [10]:
untranslated = set(products["product_category_name"].dropna()) - set(translation["product_category_name"])
print("categories without translation:", len(untranslated))
sorted(untranslated)

categories without translation: 2


['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

## My findings (write these yourself)

Fill in 1 to 2 lines for each. These become your cleaning decisions on Day 6 and the data-quality section of the README.

1. **Biggest and smallest tables:**
2. **Columns with many nulls, and why I think they are null:**
3. **Any key that was not unique, and what the repeated rows are:**
4. **Any orphan rows between tables:**
5. **Date range of the data:**
6. **Anything odd in order status or dates:**
7. **What I plan to clean, and how:**